# Damped and forced harmonic oscillator

Author: Kevin Schäfers, kschaefers@uni-wuppertal.de

Date: 08.04.2026

---------------------------------------------------

This Jupyter notebook implements the hierarchical splitting methods used in 

*K. Schäfers, M. Günther: A hierarchical splitting approach for N-split ordinary differential equations*

to perform numerical simulations for a damped and forced harmonic oscillator. Moreover, this Jupyter notebook contains the scripts with which the numerical results have been obtained. For the numerical results reported in the paper, we have executed this Jupyter notebook with Python version 3.13.0 on a MacBook Pro 2021 with M1 Pro chip.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from scipy.integrate import solve_ivp
from scipy.linalg import expm
#import matplot2tikz # used to export the matplotlib figures to TikZ for LaTeX

In [ ]:
omega = 5 # frequency of the oscillator
d = 0.1 # damping
m = 0.4 # mass
u = lambda t: 5*np.sin(t) # input

J = np.array([[0,-1],[1,0]]) # structure matrix
R = np.array([[d,0],[0,0]]) # dissipation matrix
B = np.array([1/m,0]) # port matrix
Q = np.array([[1,0],[0,omega**2]]) # matrix of the Hamiltonian H = 0.5 x^T Q x

def commutator(A,B):
    """
    matrix commutator [A,B] = AB - BA
    """

    return A@B - B@A

C = commutator(R@Q , commutator(R@Q,J@Q)) # force-gradient term

x0 = np.array([0,np.pi/4]) # initial values
tspan = [0,40] # time interval

In [ ]:
def f(t,x):
    """
    full right-hand side of the damped and forced harmonic oscillator in port-Hamiltonian form
    """
    return (J-R)@Q@x + B*u(t)

In [ ]:
# compute reference solution with very high accuracy
sol = solve_ivp(f,tspan,x0,method='RK45',rtol=1e-12,atol=1e-14)
x_ref = sol.y[:,-1] 

In [ ]:
def BlanesMoan2002(x0,tspan,nsteps):
    """
    Fourth-order composition method with s=6 stages from Blanes and Moan (2002) that uses the Lie-Trotter splitting for N-split systems as the underlying base scheme.
    The method splits the vector field into four parts: the conservative part, the dissipative part, the input part,
    and the auxiliary ODE for the time t.

    Input:
    x0 : initial value 
    tspan : time interval [t_0,t_end]
    nsteps : number of steps on an equidistant time grid

    Output:
    t : time points 
    x : numerical solution, x[:,n] is the solution at time t[n]
    """
    h = (tspan[1]-tspan[0])/nsteps
    t = np.zeros(nsteps+1)
    t[0] = tspan[0]
    x = np.zeros((len(x0),nsteps+1))
    x[:,0] = x0

    c = np.array([
        0.0829844064174052,
        0.233995250731498,
        -0.409933719901930,
        0.059762097006575,
        0.370877414979582,
        0.162314550766870
    ])
    s = len(c)
    
    for n in range(nsteps):
        x[:,n+1] = x[:,n]
        t[n+1] = t[n]
        for i in range(s):
            t[n+1] += c[i]*h
            x[:,n+1] += c[i]*h*B*u(t[n+1])
            x[0,n+1] = np.exp(-d*(c[i]*h)) * x[0,n+1]
            x[:,n+1] = np.array([[np.cos(omega*(c[i]+c[s-1-i])*h),-omega*np.sin(omega*(c[i]+c[s-1-i])*h)],[np.sin(omega*(c[i]+c[s-1-i])*h)/omega,np.cos(omega*(c[i]+c[s-1-i])*h)]])@x[:,n+1]
            x[0,n+1] = np.exp(-d*(c[s-1-i]*h)) * x[0,n+1]
            x[:,n+1] += c[s-1-i]*h*B*u(t[n+1])
            t[n+1] += c[s-1-i]*h
       
    return t,x

In [ ]:
def HSM(x0,tspan,nsteps):
    """
    Hierarchical splitting method that is based on the force-gradient integrators from [Moench and Marheineke (2025)] 
    for linear port-Hamiltonian systems.
    The method splits the vector field into four parts: the conservative part, the dissipative part, the input part,
    and the auxiliary ODE for the time t.

    Input:
    x0 : initial value 
    tspan : time interval [t_0,t_end]
    nsteps : number of steps on an equidistant time grid

    Output:
    t : time points 
    x : numerical solution, x[:,n] is the solution at time t[n]
    """

    h = (tspan[1]-tspan[0])/nsteps # step size 
    t = np.zeros(nsteps+1)
    t[0] = tspan[0]
    x = np.zeros((len(x0),nsteps+1))
    x[:,0] = x0

    mu = (1 - h**2/(4*72) * d**2)*omega # parameter used to compute the matrix exponential of the conservative part including the force-gradient term

    for n in range(nsteps):
        x[:,n+1] = x[:,n]

        x[0,n+1] = x[0,n+1] + (h/6)/m * u(t[n]) # flow of the input part

        t[n+1] = t[n] + h/2 # flow of the auxiliary ODE for the time t
        x[0,n+1] = np.exp(-d*(h/12)) * x[0,n+1] # flow of the dissipative part
        x[:,n+1] = np.array([[x[0,n+1],-omega*x[1,n+1]],[x[1,n+1],x[0,n+1]/omega]])@np.array([np.cos(mu*(h/4)),np.sin(mu*(h/4))]) # flow of the conservative part (incl. the force-gradient term C)
        x[0,n+1] = np.exp(-d*(h/3)) * x[0,n+1]
        x[:,n+1] = np.array([[x[0,n+1],-omega*x[1,n+1]],[x[1,n+1],x[0,n+1]/omega]])@np.array([np.cos(mu*(h/4)),np.sin(mu*(h/4))])
        x[0,n+1] = np.exp(-d*(h/12)) * x[0,n+1]

        x[0,n+1] = x[0,n+1] + (2*h/3)/m * u(t[n+1])

        t[n+1] += h/2
        x[0,n+1] = np.exp(-d*(h/12)) * x[0,n+1]
        x[:,n+1] = np.array([[x[0,n+1],-omega*x[1,n+1]],[x[1,n+1],x[0,n+1]/omega]])@np.array([np.cos(mu*(h/4)),np.sin(mu*(h/4))])
        x[0,n+1] = np.exp(-d*(h/3)) * x[0,n+1]
        x[:,n+1] = np.array([[x[0,n+1],-omega*x[1,n+1]],[x[1,n+1],x[0,n+1]/omega]])@np.array([np.cos(mu*(h/4)),np.sin(mu*(h/4))])
        x[0,n+1] = np.exp(-d*(h/12)) * x[0,n+1]

        x[0,n+1] = x[0,n+1] + (h/6)/m * u(t[n+1])

    return t,x

In [ ]:
#t_ref,x_ref = BlanesMoan2002(x0,tspan,tspan[1]*2**11)
#x_ref = x_ref[:,-1]

In [ ]:
NIT = 20 # number of iterations for the CPU time measurements

nstep_array = np.array([2**k for k in range(7,12)])
global_error = np.zeros((2,len(nstep_array)))
CPU_times = np.zeros((2,len(nstep_array)))

for i in range(len(nstep_array)):
    for k in range(NIT):
        start = time.process_time()
        t,x = HSM(x0,tspan,2*nstep_array[i])
        end = time.process_time()
        CPU_times[0,i] += end-start
    CPU_times[0,i] /= NIT
    global_error[0,i] = np.linalg.norm(x_ref - x[:,-1])

    for k in range(NIT):
        start = time.process_time()
        t,x = BlanesMoan2002(x0,tspan,nstep_array[i])
        end = time.process_time()
        CPU_times[1,i] += end-start
    CPU_times[1,i] /= NIT    
    global_error[1,i] = np.linalg.norm(x_ref - x[:,-1])

# creating the work-precision diagram
plt.loglog(CPU_times[0,:],global_error[0,:],'-o',label='HSM')
plt.loglog(CPU_times[1,:],global_error[1,:],'-o',label=r'$BM_6 4$')
plt.legend()
plt.xlabel('CPU time [s]')
plt.ylabel('Global Error')
plt.show()
#matplot2tikz.save('PHS_WP-diagram.tex')

In [ ]:
tspan = [0,40]

def H(x):
    """
    Hamiltonian of the damped and forced harmonic oscillator, H(x) = 0.5 x^T Q x
    """
    n = len(x[0,:])
    res = np.zeros(n)
    for i in range(n):
        res[i] = 0.5*np.transpose(x[:,i])@Q@x[:,i]
    return res 

u = lambda t: 0 # for the investigations, we neglect the external input

# perform numerical simulations with both methods and large step size h = 2.5
t1,x1 = HSM(x0,tspan,16)
t2,x2 = BlanesMoan2002(x0,tspan,16)

H1 = H(x1)
DELTA_H1 = np.diff(H1)
H2 = H(x2) 
DELTA_H2 = np.diff(H2)

plt.plot(t1,H1,label='HSM')
plt.plot(t2,H2,label=r'$BM_6 4$')
plt.legend() 
plt.xlabel(r'$t$')
plt.ylabel(r'$\mathcal{H}(x(t))$')
plt.show()
#matplot2tikz.save('PHS_Hamiltonian.tex')